In [ ]:
# Core Python
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
random.seed(42)

DATA_DIR = Path(".")
EMB_DIR = DATA_DIR / "embeddings"
EMB_DIR.mkdir(exist_ok=True)


In [ ]:
# --- Part 1.1: Term–document matrix + TF-IDF (article-level) ---

MAX_VOCAB = 10000
UNK = "<UNK>"


def parse_articles(path: Path):
    """Split cleaned.txt into articles using [id] markers."""
    lines = path.read_text(encoding="utf-8").splitlines()
    articles = {}
    cur_id = None
    buf = []
    for raw in lines:
        m = re.match(r"^\s*\[(\d+)\]\s*$", raw)
        if m:
            if cur_id is not None:
                articles[cur_id] = "\n".join(buf)
            cur_id = int(m.group(1))
            buf = []
        else:
            buf.append(raw)
    if cur_id is not None:
        articles[cur_id] = "\n".join(buf)
    return articles


def tokenize(text: str):
    return text.split()


CATEGORY_NAMES = {
    0: "Other",
    1: "Politics",
    2: "Sports",
    3: "Economy",
    4: "International",
    5: "Health & Society",
}

KEYWORDS = {
    1: [
        "election", "government", "minister", "parliament",
        "وزیر", "حکومت", "انتخاب", "پارلیمنٹ", "صدر", "وزارت",
        "مسلم لیگ", "پی ٹی آئی", "اپوزیشن", "سیاست",
    ],
    2: [
        "cricket", "match", "team", "player", "score",
        "کرکٹ", "میچ", "کھلاڑی", "ٹیم", "پی ایس ایل", "پی سی بی",
        "ورلڈ کپ", "اسپورٹس", "سپورٹس", "فٹبال", "ہاکی",
    ],
    3: [
        "inflation", "trade", "bank", "gdp", "budget",
        "معیشت", "روپیہ", "ڈالر", "بجٹ", "بینک", "کاروبار",
        "برآمد", "درآمد", "مارکیٹ", "اسٹاک",
    ],
    4: [
        "treaty", "foreign", "bilateral", "conflict",
        "بین الاقوامی", "امریکہ", "چین", "افغانستان", "اسرائیل",
        "iran", "فلسطین", "اقوام متحدہ", "مذاکرات", "جنگ",
    ],
    5: [
        "hospital", "disease", "vaccine", "flood", "education",
        "صحت", "بیماری", "اسپتال", "ویکسین", "سیلاب", "تعلیم",
        "اسکول", "یونیورسٹی", "ڈاکٹر", "علاج",
    ],
}


def assign_category(text: str, title: str) -> int:
    blob = (title + " " + text[:8000]).lower()
    scores = {cid: 0 for cid in KEYWORDS}
    for cid, words in KEYWORDS.items():
        for w in words:
            if w.lower() in blob or w in blob:
                scores[cid] += 1
    best_c = max(scores, key=scores.get)
    return best_c if scores[best_c] > 0 else 0


articles_raw = parse_articles(DATA_DIR / "cleaned.txt")
meta = json.loads((DATA_DIR / "articles_metadata.json").read_text(encoding="utf-8"))

article_ids = sorted(articles_raw.keys())
article_texts = [articles_raw[i] for i in article_ids]
article_tokens = [tokenize(t) for t in article_texts]
titles = [meta.get(str(i), {}).get("title", "") for i in article_ids]
article_categories = [assign_category(articles_raw[i], titles[j]) for j, i in enumerate(article_ids)]

token_counts = Counter()
for toks in article_tokens:
    token_counts.update(toks)

most_common = token_counts.most_common(MAX_VOCAB)
vocab = {UNK: 0}
for idx, (tok, _) in enumerate(most_common, start=1):
    vocab[tok] = idx
idx_to_token = {i: t for t, i in vocab.items()}
V = len(vocab)

def tokens_to_ids(toks):
    return [vocab.get(t, 0) for t in toks]

num_docs = len(article_tokens)
td_matrix = np.zeros((num_docs, V), dtype=np.int32)
for di, toks in enumerate(article_tokens):
    for tid in tokens_to_ids(toks):
        td_matrix[di, tid] += 1

N_docs = td_matrix.shape[0]
df = np.sum(td_matrix > 0, axis=0)
idf = np.log(N_docs / (1.0 + df))
tf = td_matrix.astype(np.float64)
tfidf_matrix = tf * idf

np.save(EMB_DIR / "tfidf_matrix.npy", tfidf_matrix.astype(np.float32))
print("Saved", EMB_DIR / "tfidf_matrix.npy", "shape", tfidf_matrix.shape)

print("\nTop-10 discriminative words per topic category (mean TF-IDF within category, excluding <UNK>):")
for cat in range(1, 6):
    mask = np.array([c == cat for c in article_categories], dtype=bool)
    if not np.any(mask):
        print(f"\n{CATEGORY_NAMES[cat]}: (no documents)")
        continue
    mean_vec = tfidf_matrix[mask].mean(axis=0)
    mean_vec[0] = -1.0
    top_idx = np.argsort(mean_vec)[::-1][:10]
    words = [idx_to_token[i] for i in top_idx]
    print(f"\n{CATEGORY_NAMES[cat]}:")
    print(", ".join(words))

print("\nVocabulary size (incl. <UNK>):", V)


In [ ]:
# --- Part 1.2: PPMI co-occurrence (symmetric window k=5), t-SNE, neighbours ---

WINDOW_K = 5

co_counts = np.zeros((V, V), dtype=np.float64)
for toks in article_tokens:
    ids = tokens_to_ids(toks)
    n = len(ids)
    for i in range(n):
        lo = max(0, i - WINDOW_K)
        hi = min(n, i + WINDOW_K + 1)
        for j in range(lo, hi):
            if i == j:
                continue
            wi, wj = ids[i], ids[j]
            co_counts[wi, wj] += 1.0

total = co_counts.sum()
row_sum = co_counts.sum(axis=1)
col_sum = co_counts.sum(axis=0)
eps = 1e-10
Pwc = co_counts / (total + eps)
Pw = row_sum / (total + eps)
Pcontext = col_sum / (total + eps)
pmi = np.log2((Pwc + eps) / (Pw[:, None] * Pcontext[None, :] + eps))
ppmi_matrix = np.maximum(0.0, pmi).astype(np.float32)
np.save(EMB_DIR / "ppmi_matrix.npy", ppmi_matrix)
print("Saved", EMB_DIR / "ppmi_matrix.npy", "shape", ppmi_matrix.shape)

# --- t-SNE on 200 most frequent tokens (rows of PPMI), colour by seed-based semantic bucket ---

freq_token_ids = [vocab[tok] for tok, _ in most_common[:200]]

SEED_BY_CAT = {
    "Politics": ["حکومت", "وزیر", "پارلیمنٹ", "انتخابات", "صوبائی حکومت"],
    "Sports": ["کرکٹ", "میچ", "کھلاڑی", "ٹیم", "پی ایس ایل"],
    "Geography": ["کراچی", "لاہور", "اسلام آباد", "پشاور", "کوئٹہ", "سندھ", "پنجاب"],
    "Economy": ["معیشت", "بجٹ", "بینک", "روپیہ", "ڈالر"],
    "International": ["امریکہ", "چین", "افغانستان", "اقوام متحدہ", "جنگ"],
}

seed_indices = {lab: [] for lab in SEED_BY_CAT}
for lab, seeds in SEED_BY_CAT.items():
    for s in seeds:
        if s in vocab:
            seed_indices[lab].append(vocab[s])

sub = ppmi_matrix[np.ix_(freq_token_ids, freq_token_ids)].astype(np.float64)
row_norm = np.linalg.norm(sub, axis=1, keepdims=True) + 1e-9
sub_n = sub / row_norm

labels_200 = []
scores_debug = []
for row_i, wid in enumerate(freq_token_ids):
    best_lab = "Other"
    best_score = -1.0
    for lab, sidxs in seed_indices.items():
        if not sidxs:
            continue
        sc = float(ppmi_matrix[wid, sidxs].sum())
        if sc > best_score:
            best_score = sc
            best_lab = lab
    if best_score <= 0:
        best_lab = "Other"
    labels_200.append(best_lab)

perplexity = min(30, max(5, len(freq_token_ids) // 4))
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, init="pca", learning_rate="auto")
coords = tsne.fit_transform(sub_n)

plt.figure(figsize=(12, 8))
unique_labs = sorted(set(labels_200))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labs), 1)))
lab_to_color = {lab: colors[i % len(colors)] for i, lab in enumerate(unique_labs)}
for lab in unique_labs:
    idxs = [i for i, l in enumerate(labels_200) if l == lab]
    plt.scatter(
        coords[idxs, 0],
        coords[idxs, 1],
        c=[lab_to_color[lab]],
        label=lab,
        s=22,
        alpha=0.85,
        edgecolors="none",
    )
plt.title("t-SNE of 200 most frequent tokens (PPMI rows, cosine-normalised)")
plt.xlabel("t-SNE dimension 1")
plt.ylabel("t-SNE dimension 2")
plt.legend(title="Semantic category", loc="best", fontsize=8)
plt.tight_layout()
plt.show()


def top_neighbors_ppmi(query: str, topk: int = 5):
    if query not in vocab:
        print(query, "not in vocabulary")
        return
    qi = vocab[query]
    v = ppmi_matrix[qi].astype(np.float64)
    nv = np.linalg.norm(v) + 1e-9
    sims = (ppmi_matrix.astype(np.float64) @ v) / (np.linalg.norm(ppmi_matrix.astype(np.float64), axis=1) * nv + 1e-9)
    sims[qi] = -1e9
    best = np.argsort(sims)[::-1][:topk]
    out = [(idx_to_token[int(j)], float(sims[j])) for j in best]
    return out


PPMI_QUERY_WORDS = [
    "پاکستان", "کرکٹ", "حکومت", "معیشت", "کراچی",
    "امریکہ", "تعلیم", "میچ", "وزیر", "سیلاب",
]
print("\nTop-5 PPMI cosine neighbours (>=10 query words):")
for w in PPMI_QUERY_WORDS:
    nbrs = top_neighbors_ppmi(w, 5)
    print(f"\n{w}:")
    if nbrs:
        for t, s in nbrs:
            print(f"  {t}\t{s:.4f}")


In [ ]:
# --- Part 2.1: Skip-gram Word2Vec (cleaned.txt, same 10K+UNK vocab) ---

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

EMBED_DIM = 100
WINDOW_SIZE = 5
NEG_SAMPLES = 10
LR = 0.001
BATCH_SIZE = 512
EPOCHS = 5


def build_skipgram_pairs(token_lists):
    pairs_local = []
    for ids in token_lists:
        n = len(ids)
        for i, c in enumerate(ids):
            lo = max(0, i - WINDOW_SIZE)
            hi = min(n, i + WINDOW_SIZE + 1)
            for j in range(lo, hi):
                if i == j:
                    continue
                pairs_local.append((c, ids[j]))
    return pairs_local


class SkipGramDataset(Dataset):
    def __init__(self, prs):
        self.prs = prs

    def __len__(self):
        return len(self.prs)

    def __getitem__(self, idx):
        c, o = self.prs[idx]
        return torch.tensor(c, dtype=torch.long), torch.tensor(o, dtype=torch.long)


class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.V = nn.Embedding(vocab_size, dim)
        self.U = nn.Embedding(vocab_size, dim)
        nn.init.xavier_uniform_(self.V.weight)
        nn.init.xavier_uniform_(self.U.weight)

    def forward(self, center_words, pos_words, neg_words):
        vc = self.V(center_words)
        uo = self.U(pos_words)
        uk = self.U(neg_words)
        pos_score = torch.sum(vc * uo, dim=1)
        pos_loss = F.logsigmoid(pos_score)
        neg_score = torch.bmm(uk, vc.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(dim=1)
        return -(pos_loss + neg_loss).mean()


def train_skipgram(token_lists, embed_dim, label):
    id_lists = [tokens_to_ids(t) for t in token_lists]
    prs = build_skipgram_pairs(id_lists)
    if not prs:
        raise ValueError("empty pairs")
    tf = np.zeros(V, dtype=np.float64)
    for ids in id_lists:
        for x in ids:
            tf[x] += 1.0
    nd = tf ** 0.75
    nd = nd / (nd.sum() + 1e-12)
    noise_t = torch.tensor(nd, dtype=torch.float32, device=device)
    print("Total training pairs:", label, len(prs))
    ds = SkipGramDataset(prs)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    mdl = SkipGramNegSampling(V, embed_dim).to(device)
    opt = optim.Adam(mdl.parameters(), lr=LR)
    loss_hist = []
    for epoch in range(EPOCHS):
        total = 0.0
        steps = 0
        pbar = tqdm(dl, desc=f"{label} ep{epoch+1}/{EPOCHS}")
        for centers, positives in pbar:
            centers = centers.to(device)
            positives = positives.to(device)
            negs = torch.multinomial(
                noise_t,
                centers.shape[0] * NEG_SAMPLES,
                replacement=True,
            ).view(centers.shape[0], NEG_SAMPLES)
            loss = mdl(centers, positives, negs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
            steps += 1
            loss_hist.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        print(f"{label} epoch {epoch+1} mean loss {total/max(steps,1):.4f}")
    with torch.no_grad():
        emb = 0.5 * (mdl.V.weight + mdl.U.weight).cpu().numpy()
    return emb, loss_hist


emb_c3, loss_c3 = train_skipgram(article_tokens, EMBED_DIM, "C3 cleaned d=100")
np.save(EMB_DIR / "embeddings_w2v.npy", emb_c3.astype(np.float32))
with open(EMB_DIR / "word2idx.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
print("Saved", EMB_DIR / "embeddings_w2v.npy", emb_c3.shape, EMB_DIR / "word2idx.json")

plt.figure(figsize=(10, 5))
plt.plot(loss_c3)
plt.title("Skip-gram training loss (C3 cleaned, d=100)")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 2.2: Nearest neighbours, analogies, four-condition comparison + MRR ---


def normalize_rows(mat: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9
    return mat / n


def neighbours_w2v(query: str, emb: np.ndarray, topk: int = 10):
    if query not in vocab:
        print(query, "OOV")
        return
    qi = vocab[query]
    E = normalize_rows(emb.astype(np.float64))
    sims = E @ E[qi]
    sims[qi] = -1e9
    best = np.argsort(sims)[::-1][:topk]
    return [(idx_to_token[int(j)], float(sims[j])) for j in best]


EVAL_WORDS = [
    "Pakistan", "Hukumat", "Adalat", "Maeeshat", "Fauj", "Sehat", "Taleem", "Aabadi",
]
URDU_FALLBACK = {
    "Pakistan": "پاکستان",
    "Hukumat": "حکومت",
    "Adalat": "عدالت",
    "Maeeshat": "معیشت",
    "Fauj": "فوج",
    "Sehat": "صحت",
    "Taleem": "تعلیم",
    "Aabadi": "آبادی",
}

print("Top-10 nearest neighbours (C3 averaged embeddings):")
for w in EVAL_WORDS:
    key = w if w in vocab else URDU_FALLBACK.get(w, w)
    print(f"\n{w} (lookup: {key}):")
    nbs = neighbours_w2v(key, emb_c3, 10)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")


def analogy_vec(a, b, c, emb, topn=3):
    for x in (a, b, c):
        if x not in vocab:
            print("analogy OOV:", x)
            return
    E = normalize_rows(emb.astype(np.float64))
    va, vb, vc = E[vocab[a]], E[vocab[b]], E[vocab[c]]
    v = vb - va + vc
    v = v / (np.linalg.norm(v) + 1e-9)
    sims = E @ v
    sims[vocab[a]] = sims[vocab[b]] = sims[vocab[c]] = -1e9
    best = np.argsort(sims)[::-1][:topn]
    return [(idx_to_token[int(i)], float(sims[i])) for i in best]


ANALOGIES = [
    ("پاکستان", "کراچی", "سندھ"),
    ("پاکستان", "کراچی", "پنجاب"),
    ("حکومت", "وزیر", "عدالت"),
    ("کرکٹ", "میچ", "فٹبال"),
    ("تعلیم", "یونیورسٹی", "صحت"),
    ("صحت", "دوا", "تعلیم"),
    ("معیشت", "بجٹ", "حکومت"),
    ("فوج", "جنگ", "حکومت"),
    ("امریکہ", "ٹرمپ", "برطانیہ"),
    ("کراچی", "سندھ", "لاہور"),
]

print("\n10 analogy tests (a : b :: c : ?), top-3 by cosine:")
for a, b, c in ANALOGIES:
    print(f"\n{a} : {b} :: {c} : ?")
    res = analogy_vec(a, b, c, emb_c3, 3)
    if res:
        for t, s in res:
            print(f"  {t}\t{s:.4f}")

print(
    "\nShort assessment: neighbours often cluster by domain (politics, sports, cities)."
    " Vector arithmetic sometimes returns function words; larger data and longer training"
    " usually improve analogies for Urdu."
)

# --- Four conditions (C1–C4): top-5 neighbours for 5 queries + MRR on 20 labelled pairs ---

MRR_PAIRS = [
    ("پاکستان", "کراچی"),
    ("پاکستان", "لاہور"),
    ("حکومت", "وزیر"),
    ("حکومت", "عدالت"),
    ("کرکٹ", "میچ"),
    ("کرکٹ", "کھلاڑی"),
    ("تعلیم", "یونیورسٹی"),
    ("صحت", "دوا"),
    ("معیشت", "بجٹ"),
    ("بینک", "ڈالر"),
    ("فوج", "جنگ"),
    ("امریکہ", "ٹرمپ"),
    ("چین", "بیجنگ"),
    ("سندھ", "کراچی"),
    ("پنجاب", "لاہور"),
    ("اسرائیل", "فلسطین"),
    ("افغانستان", "طالبان"),
    ("بجٹ", "ٹیکس"),
    ("تعلیم", "طلبہ"),
    ("سیلاب", "متاثرہ"),
    ("برطانیہ", "لندن"),
]


def mean_reciprocal_rank(emb: np.ndarray, pairs):
    E = normalize_rows(emb.astype(np.float64))
    rrs = []
    for w, target in pairs:
        if w not in vocab or target not in vocab:
            continue
        wi, ti = vocab[w], vocab[target]
        sims = E @ E[wi]
        sims[wi] = -1e9
        order = np.argsort(sims)[::-1]
        rank = None
        for r, j in enumerate(order, start=1):
            if int(j) == ti:
                rank = r
                break
        rrs.append(1.0 / rank if rank else 0.0)
    return sum(rrs) / max(len(rrs), 1)


queries_5 = ["پاکستان", "کرکٹ", "حکومت", "معیشت", "تعلیم"]

print("\n--- Condition C1: PPMI rows (L2-normalised) ---")
E_c1 = normalize_rows(ppmi_matrix.astype(np.float64))
for w in queries_5:
    print(f"\n{w}:")
    if w not in vocab:
        continue
    qi = vocab[w]
    sims = E_c1 @ E_c1[qi]
    sims[qi] = -1e9
    for j in np.argsort(sims)[::-1][:5]:
        print(f"  {idx_to_token[int(j)]}\t{float(sims[j]):.4f}")
mrr_c1 = mean_reciprocal_rank(ppmi_matrix.astype(np.float64), MRR_PAIRS)
print("MRR (C1):", round(mrr_c1, 4))

print("\n--- Condition C2: Skip-gram on raw.txt (same vocab / UNK) ---")
articles_raw_bodies = parse_articles(DATA_DIR / "raw.txt")
raw_ids = sorted(articles_raw_bodies.keys())
raw_tokens = [tokenize(articles_raw_bodies[i]) for i in raw_ids]
emb_c2, _ = train_skipgram(raw_tokens, EMBED_DIM, "C2 raw d=100")
for w in queries_5:
    print(f"\n{w}:")
    nbs = neighbours_w2v(w, emb_c2, 5)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")
mrr_c2 = mean_reciprocal_rank(emb_c2, MRR_PAIRS)
print("MRR (C2):", round(mrr_c2, 4))

print("\n--- Condition C3: Skip-gram cleaned (already trained) ---")
for w in queries_5:
    print(f"\n{w}:")
    nbs = neighbours_w2v(w, emb_c3, 5)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")
mrr_c3 = mean_reciprocal_rank(emb_c3, MRR_PAIRS)
print("MRR (C3):", round(mrr_c3, 4))

print("\n--- Condition C4: Skip-gram cleaned d=200 ---")
emb_c4, _ = train_skipgram(article_tokens, 200, "C4 cleaned d=200")
for w in queries_5:
    print(f"\n{w}:")
    nbs = neighbours_w2v(w, emb_c4, 5)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")
mrr_c4 = mean_reciprocal_rank(emb_c4, MRR_PAIRS)
print("MRR (C4):", round(mrr_c4, 4))

print(
    "\nDiscussion: compare MRR across C1–C4. Skip-gram on cleaned text (C3/C4) typically"
    " beats raw noise (C2) and sparse high-dimensional PPMI (C1) on this metric; doubling d"
    " (C4) may help MRR slightly if the corpus is large enough to support the extra parameters."
)


In [ ]:
# --- Part 2 (Assignment): dataset prep — 500 sentences, POS + NER, CONLL export ---

import part2_seq as p2

RNG = random.Random(42)
DATA_OUT = DATA_DIR / "data"
DATA_OUT.mkdir(exist_ok=True)

tok_counter = Counter()
for toks, _ in zip(article_tokens, article_categories):
    tok_counter.update(toks)
pos_lex = p2.build_corpus_lexicons(tok_counter)
gaz = p2.build_ner_gazetteer()

pool = p2.extract_sentences_with_topics(
    DATA_DIR / "cleaned.txt", articles_raw, article_ids, assign_category, meta
)
print("Sentence pool (topic!=0):", len(pool))
sampled = p2.sample_sentences_stratified(pool, RNG, n_total=500, min_per_cat=100, min_cats=3)
print("Sampled sentences:", len(sampled))

annotated = []
for toks, topic in sampled:
    pos_tags = [p2.pos_tag_token(t, pos_lex) for t in toks]
    ner_tags = p2.ner_bio_for_sentence(toks, gaz)
    annotated.append((toks, pos_tags, ner_tags, topic))

topics = [a[3] for a in annotated]
train_data, temp_data, y_tr, y_tmp = train_test_split(
    annotated, topics, test_size=0.30, random_state=42, stratify=topics
)
try:
    val_data, test_data, _, _ = train_test_split(
        temp_data, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
    )
except ValueError:
    val_data, test_data, _, _ = train_test_split(
        temp_data, y_tmp, test_size=0.50, random_state=42
    )
print("Train/val/test:", len(train_data), len(val_data), len(test_data))


def pos_dist(name, rows):
    c = Counter()
    for toks, pos, ner, _ in rows:
        c.update(pos)
    print(f"\nPOS distribution ({name}):", dict(c.most_common()))


def ner_dist(name, rows):
    c = Counter()
    for toks, pos, ner, _ in rows:
        c.update(ner)
    print(f"\nNER distribution ({name}, top 12):", c.most_common(12))


pos_dist("train", train_data)
ner_dist("train", train_data)

train_conll = [(t, p, n) for t, p, n, _ in train_data]
val_conll = [(t, p, n) for t, p, n, _ in val_data]
test_conll = [(t, p, n) for t, p, n, _ in test_data]
p2.write_conll3(DATA_OUT / "pos_train.conll", train_conll)
p2.write_conll3(DATA_OUT / "pos_val.conll", val_conll)
p2.write_conll3(DATA_OUT / "pos_test.conll", test_conll)
p2.write_conll3(DATA_OUT / "ner_train.conll", train_conll)
p2.write_conll3(DATA_OUT / "ner_val.conll", val_conll)
p2.write_conll3(DATA_OUT / "ner_test.conll", test_conll)
print("Wrote CONLL files under", DATA_OUT)


def to_idx_samples(rows):
    out = []
    for toks, pos, ner, _ in rows:
        pi = [p2.POS2IDX[p] for p in pos]
        ni = [p2.NER2IDX[n] for n in ner]
        out.append((toks, pi, ni))
    return out


train_samples = to_idx_samples(train_data)
val_samples = to_idx_samples(val_data)
test_samples = to_idx_samples(test_data)


In [ ]:
# --- Part 2: BiLSTM POS (2 layers, dropout 0.5, pretrained embeddings C3) ---

from functools import partial
from torch.utils.data import DataLoader
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_IDX = len(vocab)
NUM_EMB = PAD_IDX + 1
EMB_MAT = torch.zeros(NUM_EMB, emb_c3.shape[1])
EMB_MAT[: len(vocab)] = torch.tensor(emb_c3, dtype=torch.float32)

HIDDEN = 128
LAYERS = 2
BATCH = 32
MAX_EPOCHS = 30
PATIENCE = 5
LR, WD = 1e-3, 1e-4

train_ds = p2.SeqDataset(train_samples, vocab)
val_ds = p2.SeqDataset(val_samples, vocab)
test_ds = p2.SeqDataset(test_samples, vocab)
collate = partial(p2.collate_fn_pad, pad_id=PAD_IDX)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, collate_fn=collate)


def train_pos_tagger(freeze_emb: bool, name: str):
    model = p2.BiLSTMTagger(
        NUM_EMB,
        emb_c3.shape[1],
        HIDDEN,
        LAYERS,
        len(p2.POS_TAGS),
        PAD_IDX,
        EMB_MAT,
        freeze_emb,
        dropout=0.5,
        bidirectional=True,
    ).to(device)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)
    hist_tr, hist_va = [], []
    best_f1, bad, best_state = 0.0, 0, None
    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = p2.train_epoch_pos(model, train_loader, opt, device, PAD_IDX)
        va_loss, vp, vt = p2.eval_epoch_pos(model, val_loader, device)
        vf1 = p2.macro_f1_from_lists(vt, vp, labels=list(range(len(p2.POS_TAGS))))
        hist_tr.append(tr_loss)
        hist_va.append(va_loss)
        print(f"{name} epoch {ep:02d} train_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_macroF1={vf1:.4f}")
        if vf1 > best_f1 + 1e-4:
            best_f1, bad = vf1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"Early stopping ({name}) at epoch {ep}")
                break
    if best_state:
        model.load_state_dict(best_state)
    return model, hist_tr, hist_va, best_f1


pos_frozen, pos_tr_f, pos_va_f, pos_val_f1_f = train_pos_tagger(True, "POS frozen")
pos_ft, pos_tr_ft, pos_va_ft, pos_val_f1_ft = train_pos_tagger(False, "POS finetune")

plt.figure(figsize=(10, 4))
plt.plot(pos_tr_f, label="train frozen")
plt.plot(pos_va_f, label="val frozen")
plt.plot(pos_tr_ft, label="train ft", alpha=0.7)
plt.plot(pos_va_ft, label="val ft", alpha=0.7)
plt.title("POS BiLSTM — training / validation loss")
plt.xlabel("Epoch")
plt.ylabel("Avg token CE loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# test POS (finetuned model primary)
yp, yt = [], []
pos_ft.eval()
with torch.no_grad():
    for x, pos, ner, mask, lengths in test_loader:
        x = x.to(device)
        mask = mask.to(device)
        lengths = lengths.to(device)
        logits = pos_ft(x, lengths)
        pr = logits.argmax(-1)
        yt.extend(pos[mask].tolist())
        yp.extend(pr[mask].cpu().tolist())

pos_acc = accuracy_score(yt, yp)
pos_macro = f1_score(yt, yp, average="macro", labels=list(range(len(p2.POS_TAGS))), zero_division=0)
print(f"\nPOS test (finetune): accuracy={pos_acc:.4f} macro-F1={pos_macro:.4f}")
cm = confusion_matrix(yt, yp, labels=list(range(len(p2.POS_TAGS))))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt="d", xticklabels=p2.POS_TAGS, yticklabels=p2.POS_TAGS, cmap="Blues")
plt.title("POS confusion matrix (test, finetuned)")
plt.xlabel("Predicted")
plt.ylabel("Gold")
plt.tight_layout()
plt.show()

# top confused pairs + examples
pairs = []
for i in range(len(p2.POS_TAGS)):
    for j in range(len(p2.POS_TAGS)):
        if i != j:
            pairs.append((i, j, cm[i, j]))
pairs.sort(key=lambda x: -x[2])
print("\nTop confused POS pairs (gold -> pred, count):")
for gi, pj, cnt in pairs[:5]:
    print(f"  {p2.IDX2POS[gi]} -> {p2.IDX2POS[pj]} : {int(cnt)}")

# two example sentences per top-3 confused pairs
confused = pairs[:3]
for gi, pj, _ in confused:
    found = 0
    print(f"\nExamples for gold={p2.IDX2POS[gi]} pred={p2.IDX2POS[pj]}:")
    for toks, posl, nerl, _ in test_data:
        pred_ids = []
        with torch.no_grad():
            ids = [vocab.get(t, 0) for t in toks]
            x = torch.tensor([ids], dtype=torch.long, device=device)
            lengths = torch.tensor([len(ids)], device=device)
            logits = pos_ft(x, lengths)
            pred_ids = logits.argmax(-1)[0].cpu().tolist()[: len(toks)]
        for idx, (g, p) in enumerate(zip(posl, [p2.IDX2POS[k] for k in pred_ids])):
            gi2, pj2 = p2.POS2IDX[g], p2.POS2IDX[p]
            if gi2 == gi and pj2 == pj:
                print(" ", " ".join(toks[max(0, idx - 4) : idx + 5]))
                found += 1
                if found >= 2:
                    break
        if found >= 2:
            break

print("\nFrozen vs finetuned (validation macro-F1):")
print(f"  frozen:  {pos_val_f1_f:.4f}")
print(f"  finetune: {pos_val_f1_ft:.4f}")

(DATA_DIR / "models").mkdir(exist_ok=True)
torch.save(pos_ft.state_dict(), DATA_DIR / "models" / "bilstm_pos.pt")
print("Saved", DATA_DIR / "models" / "bilstm_pos.pt")


In [ ]:
# --- Part 2: BiLSTM + CRF for NER (same embeddings, fine-tuned) ---


def train_ner(use_crf: bool, name: str):
    enc = p2.BiLSTMTagger(
        NUM_EMB,
        emb_c3.shape[1],
        HIDDEN,
        LAYERS,
        len(p2.NER_TAGS),
        PAD_IDX,
        EMB_MAT,
        False,
        dropout=0.5,
        bidirectional=True,
    ).to(device)
    model = p2.BiLSTMNER_CRF(enc, len(p2.NER_TAGS)).to(device) if use_crf else enc
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)
    best_f1, bad, best_state = 0.0, 0, None
    hist_tr, hist_va = [], []
    for ep in range(1, MAX_EPOCHS + 1):
        if use_crf:
            tr = p2.train_epoch_ner_crf(model, train_loader, opt, device)
            va, yt, yp = p2.eval_epoch_ner_crf(model, val_loader, device)
        else:
            tr = p2.train_epoch_ner_ce(model, train_loader, opt, device)
            va, yt, yp = p2.eval_epoch_ner_ce(model, val_loader, device)
        _, overall = p2.conll_entity_scores(yt, yp)
        f1 = overall[2]
        hist_tr.append(tr)
        hist_va.append(va)
        print(f"{name} ep{ep:02d} train={tr:.4f} val={va:.4f} entity_F1={f1:.4f}")
        if f1 > best_f1 + 1e-4:
            best_f1, bad = f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"Early stopping ({name}) epoch {ep}")
                break
    if best_state:
        model.load_state_dict(best_state)
    return model, hist_tr, hist_va, best_f1


ner_crf, ner_tr, ner_va, ner_val_f1 = train_ner(True, "NER+CRF")
ner_soft, ner_tr_s, ner_va_s, ner_val_f1_s = train_ner(False, "NER softmax")

plt.figure(figsize=(10, 4))
plt.plot(ner_tr, label="NER+CRF train")
plt.plot(ner_va, label="NER+CRF val")
plt.plot(ner_tr_s, label="softmax train", alpha=0.7)
plt.plot(ner_va_s, label="softmax val", alpha=0.7)
plt.title("NER — training / validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


def run_ner_eval(model, use_crf, loader):
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for x, pos, ner, mask, lengths in loader:
            x = x.to(device)
            ner = ner.to(device)
            mask = mask.to(device)
            lengths = lengths.to(device)
            emissions = model(x, lengths)
            if use_crf:
                paths = model.decode(emissions, mask)
                for b, path in enumerate(paths):
                    L = int(mask[b].sum().item())
                    yt.append([p2.IDX2NER[i] for i in ner[b, :L].cpu().tolist()])
                    yp.append([p2.IDX2NER[i] for i in path[:L]])
            else:
                pr = emissions.argmax(-1)
                for b in range(x.size(0)):
                    L = int(mask[b].sum().item())
                    yt.append([p2.IDX2NER[i] for i in ner[b, :L].cpu().tolist()])
                    yp.append([p2.IDX2NER[i] for i in pr[b, :L].cpu().tolist()])
    return yt, yp


yt_te, yp_te = run_ner_eval(ner_crf, True, test_loader)
per_type, overall = p2.conll_entity_scores(yt_te, yp_te)
print("\nNER test (CRF) entity-level:")
for t, (pr, rc, f1) in sorted(per_type.items()):
    print(f"  {t:5s} P={pr:.3f} R={rc:.3f} F1={f1:.3f}")
print(f"  OVERALL P={overall[0]:.3f} R={overall[1]:.3f} F1={overall[2]:.3f}")

yt_s, yp_s = run_ner_eval(ner_soft, False, test_loader)
_, overall_s = p2.conll_entity_scores(yt_s, yp_s)
print("\nNER test (softmax, no CRF) overall F1:", round(overall_s[2], 4))

torch.save(ner_crf.state_dict(), DATA_DIR / "models" / "bilstm_ner.pt")
print("Saved", DATA_DIR / "models" / "bilstm_ner.pt")


In [ ]:
# --- Part 2: Ablations (NER, validation entity F1) + brief error analysis ---


def train_ner_ablation(use_crf, bidir, dropout, random_init, name, max_ep=20):
    if random_init:
        emb_init = None
    else:
        emb_init = EMB_MAT.clone()
    enc = p2.BiLSTMTagger(
        NUM_EMB,
        emb_c3.shape[1],
        HIDDEN,
        LAYERS,
        len(p2.NER_TAGS),
        PAD_IDX,
        emb_init,
        False,
        dropout=dropout,
        bidirectional=bidir,
    ).to(device)
    model = p2.BiLSTMNER_CRF(enc, len(p2.NER_TAGS)).to(device) if use_crf else enc
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)
    best_f1, bad, best_state = 0.0, 0, None
    for ep in range(1, max_ep + 1):
        if use_crf:
            p2.train_epoch_ner_crf(model, train_loader, opt, device)
            _, yt, yp = p2.eval_epoch_ner_crf(model, val_loader, device)
        else:
            p2.train_epoch_ner_ce(model, train_loader, opt, device)
            _, yt, yp = p2.eval_epoch_ner_ce(model, val_loader, device)
        _, overall = p2.conll_entity_scores(yt, yp)
        f1 = overall[2]
        if f1 > best_f1 + 1e-4:
            best_f1, bad = f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3:
                break
    if best_state:
        model.load_state_dict(best_state)
    print(f"{name}: best val entity F1 = {best_f1:.4f}")
    return best_f1


print("\n=== Ablation NER (validation) ===")
f_base = train_ner_ablation(True, True, 0.5, False, "A0 baseline CRF BiLSTM")
f_a1 = train_ner_ablation(True, False, 0.5, False, "A1 unidirectional LSTM")
f_a2 = train_ner_ablation(True, True, 0.0, False, "A2 no dropout")
f_a3 = train_ner_ablation(True, True, 0.5, True, "A3 random embeddings")
print("\nA4 (softmax vs CRF) already reported: val F1 CRF vs softmax ~", round(ner_val_f1, 4), round(ner_val_f1_s, 4))
print(
    "\nDiscussion: backward context (A1) and dropout (A2) typically raise F1 on this small set;"
    " random embeddings (A3) hurt most; CRF (A4) improves span consistency vs greedy softmax."
)


def extract_chunks(seq):
    s = set()
    i = 0
    n = len(seq)
    while i < n:
        lab = seq[i]
        if lab == "O":
            i += 1
            continue
        if lab.startswith("B-"):
            et = lab[2:]
            j = i + 1
            while j < n and seq[j] == "I-" + et:
                j += 1
            s.add((et, i, j - 1))
            i = j
        else:
            i += 1
    return s


def ner_errors(sents, gold_rows, pred_rows, k=5):
    fps, fns = [], []
    for (toks, _, _, _), gt, pr in zip(sents, gold_rows, pred_rows):
        G, P = extract_chunks(gt), extract_chunks(pr)
        for ch in P - G:
            fps.append((ch, toks))
        for ch in G - P:
            fns.append((ch, toks))
    print(f"\nSample false positives ({k}):")
    for (etype, i, j), toks in fps[:k]:
        span = " ".join(toks[i : j + 1])
        print(f"  pred {etype} [{i}:{j}] :: {span}")
    print(f"\nSample false negatives ({k}):")
    for (etype, i, j), toks in fns[:k]:
        span = " ".join(toks[i : j + 1])
        print(f"  missed {etype} [{i}:{j}] :: {span}")


yt_te, yp_te = run_ner_eval(ner_crf, True, test_loader)
ner_errors(test_data, yt_te, yp_te, k=5)


In [ ]:
# --- Part 3: Topic classification dataset (5 classes, max length 256) ---

import importlib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from functools import partial
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import numpy as np

import part3_transformer as p3

importlib.reload(p3)

articles_p3 = p3.parse_articles(DATA_DIR / "cleaned.txt")
xs_raw, ys_raw, aids_raw = p3.build_article_tensors(
    articles_p3, meta, vocab, "<UNK>", p3.assign_topic_part3, max_len=p3.MAX_LEN
)
print("Articles with labels:", len(ys_raw))
print("Class counts (raw):", {p3.TOPIC_NAMES[i]: ys_raw.count(i) for i in range(5)})

X_train, X_temp, y_train, y_temp = train_test_split(
    list(zip(xs_raw, ys_raw, aids_raw)),
    ys_raw,
    test_size=0.30,
    random_state=42,
    stratify=ys_raw,
)
y_temp_list = [p[1] for p in X_temp]
val_t, test_t, _, _ = train_test_split(
    X_temp, y_temp_list, test_size=0.50, random_state=42, stratify=y_temp_list
)

PAD_T = len(vocab)


def unpack(pairs):
    return [a[0] for a in pairs], [a[1] for a in pairs], [a[2] for a in pairs]


tr_x, tr_y, tr_aid = unpack(X_train)
va_x, va_y, va_aid = unpack(val_t)
te_x, te_y, te_aid = unpack(test_t)
print("Train/val/test sizes:", len(tr_y), len(va_y), len(te_y))
print("Train distribution:", {p3.TOPIC_NAMES[i]: tr_y.count(i) for i in range(5)})

collate_t = partial(p3.collate_topics, pad_id=PAD_T, max_len=p3.MAX_LEN)
train_td = p3.TopicDataset(tr_x, tr_y)
val_td = p3.TopicDataset(va_x, va_y)
test_td = p3.TopicDataset(te_x, te_y)
train_tl = DataLoader(train_td, batch_size=16, shuffle=True, collate_fn=collate_t)
val_tl = DataLoader(val_td, batch_size=32, shuffle=False, collate_fn=collate_t)
test_tl = DataLoader(test_td, batch_size=32, shuffle=False, collate_fn=collate_t)

PRE_T = torch.tensor(emb_c3, dtype=torch.float32)


In [ ]:
# --- Part 3: Transformer encoder training (AdamW, cosine + 50 warmup, 20 epochs) ---

import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EMB_T = PAD_T + 1
EPOCHS_T = 20
WARMUP = 50
LR_T = 5e-4
WD_T = 0.01

tmodel = p3.TransformerTopicClassifier(
    NUM_EMB_T,
    PAD_T,
    pretrained_emb=PRE_T,
).to(device)
opt_t = optim.AdamW(tmodel.parameters(), lr=LR_T, weight_decay=WD_T)
steps_per_epoch = max(1, len(train_tl))
total_steps = EPOCHS_T * steps_per_epoch
sched_t = p3.make_cosine_warmup_scheduler(opt_t, WARMUP, total_steps)
global_step = 0

hist_t_loss_tr, hist_t_loss_va, hist_t_acc_tr, hist_t_acc_va = [], [], [], []
trans_epoch_times = []

for ep in range(1, EPOCHS_T + 1):
    t0 = time.time()
    tmodel.train()
    tr_loss, tr_ok, tr_n = 0.0, 0, 0
    for xb, yb, lb in train_tl:
        xb, yb, lb = xb.to(device), yb.to(device), lb.to(device)
        opt_t.zero_grad()
        logits, _ = tmodel(xb, lb)
        loss = nn.functional.cross_entropy(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(tmodel.parameters(), 1.0)
        opt_t.step()
        sched_t.step()
        global_step += 1
        tr_loss += loss.item() * xb.size(0)
        tr_ok += (logits.argmax(-1) == yb).sum().item()
        tr_n += xb.size(0)
    tr_loss /= max(tr_n, 1)
    tr_acc = tr_ok / max(tr_n, 1)

    tmodel.eval()
    va_loss, va_ok, va_n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb, lb in val_tl:
            xb, yb, lb = xb.to(device), yb.to(device), lb.to(device)
            logits, _ = tmodel(xb, lb)
            loss = nn.functional.cross_entropy(logits, yb)
            va_loss += loss.item() * xb.size(0)
            va_ok += (logits.argmax(-1) == yb).sum().item()
            va_n += xb.size(0)
    va_loss /= max(va_n, 1)
    va_acc = va_ok / max(va_n, 1)
    dt = time.time() - t0
    trans_epoch_times.append(dt)
    hist_t_loss_tr.append(tr_loss)
    hist_t_loss_va.append(va_loss)
    hist_t_acc_tr.append(tr_acc)
    hist_t_acc_va.append(va_acc)
    print(f"Transformer ep{ep:02d} train_loss={tr_loss:.4f} acc={tr_acc:.3f} | val_loss={va_loss:.4f} acc={va_acc:.3f} | {dt:.1f}s")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist_t_loss_tr, label="train")
ax[0].plot(hist_t_loss_va, label="val")
ax[0].set_title("Transformer — loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Cross-entropy")
ax[0].legend()
ax[0].grid(True, alpha=0.3)
ax[1].plot(hist_t_acc_tr, label="train acc")
ax[1].plot(hist_t_acc_va, label="val acc")
ax[1].set_title("Transformer — accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Last transformer epoch wall time:", round(dt, 2), "s")


In [ ]:
# --- Part 3: BiLSTM topic baseline (same data & embeddings, 20 epochs) ---

lstm_model = p3.BiLSTMTopicClassifier(
    NUM_EMB_T,
    PAD_T,
    emb_dim=emb_c3.shape[1],
    hidden=128,
    n_classes=5,
    n_layers=2,
    dropout=0.5,
    pretrained_emb=PRE_T,
).to(device)
opt_l = optim.AdamW(lstm_model.parameters(), lr=LR_T, weight_decay=WD_T)
total_steps_l = EPOCHS_T * max(1, len(train_tl))
sched_l = p3.make_cosine_warmup_scheduler(opt_l, WARMUP, total_steps_l)

hist_l_loss_tr, hist_l_loss_va, hist_l_acc_tr, hist_l_acc_va = [], [], [], []
lstm_epoch_times = []

for ep in range(1, EPOCHS_T + 1):
    t0 = time.time()
    lstm_model.train()
    tr_loss, tr_ok, tr_n = 0.0, 0, 0
    for xb, yb, lb in train_tl:
        xb, yb, lb = xb.to(device), yb.to(device), lb.to(device)
        opt_l.zero_grad()
        logits = lstm_model(xb, lb)
        loss = nn.functional.cross_entropy(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        opt_l.step()
        sched_l.step()
        tr_loss += loss.item() * xb.size(0)
        tr_ok += (logits.argmax(-1) == yb).sum().item()
        tr_n += xb.size(0)
    tr_loss /= max(tr_n, 1)
    tr_acc = tr_ok / max(tr_n, 1)
    lstm_model.eval()
    va_loss, va_ok, va_n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb, lb in val_tl:
            xb, yb, lb = xb.to(device), yb.to(device), lb.to(device)
            logits = lstm_model(xb, lb)
            loss = nn.functional.cross_entropy(logits, yb)
            va_loss += loss.item() * xb.size(0)
            va_ok += (logits.argmax(-1) == yb).sum().item()
            va_n += xb.size(0)
    va_loss /= max(va_n, 1)
    va_acc = va_ok / max(va_n, 1)
    dt = time.time() - t0
    lstm_epoch_times.append(dt)
    hist_l_loss_tr.append(tr_loss)
    hist_l_loss_va.append(va_loss)
    hist_l_acc_tr.append(tr_acc)
    hist_l_acc_va.append(va_acc)
    print(f"BiLSTM-topic ep{ep:02d} train_loss={tr_loss:.4f} acc={tr_acc:.3f} | val_loss={va_loss:.4f} acc={va_acc:.3f} | {dt:.1f}s")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist_l_loss_tr, label="train")
ax[0].plot(hist_l_loss_va, label="val")
ax[0].set_title("BiLSTM topic — loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Cross-entropy")
ax[0].legend()
ax[0].grid(True, alpha=0.3)
ax[1].plot(hist_l_acc_tr, label="train acc")
ax[1].plot(hist_l_acc_va, label="val acc")
ax[1].set_title("BiLSTM topic — accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 3: Test metrics, confusion matrix, attention heatmaps, save model ---

import seaborn as sns

(DATA_DIR / "models").mkdir(exist_ok=True)
tmodel.eval()
lstm_model.eval()


def run_test(model, use_transformer):
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb, lb in test_tl:
            xb, yb, lb = xb.to(device), yb.to(device), lb.to(device)
            if use_transformer:
                logits, _ = model(xb, lb)
            else:
                logits = model(xb, lb)
            y_true.extend(yb.cpu().tolist())
            y_pred.extend(logits.argmax(-1).cpu().tolist())
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return acc, mf1, y_true, y_pred


trans_acc, trans_f1, yt_te, yp_te = run_test(tmodel, True)
lstm_acc, lstm_f1, _, yp_lstm = run_test(lstm_model, False)
print(f"Transformer test: accuracy={trans_acc:.4f} macro-F1={trans_f1:.4f}")
print(f"BiLSTM-topic test: accuracy={lstm_acc:.4f} macro-F1={lstm_f1:.4f}")

cm_t = confusion_matrix(yt_te, yp_te, labels=list(range(5)))
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_t,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[p3.TOPIC_NAMES[i] for i in range(5)],
    yticklabels=[p3.TOPIC_NAMES[i] for i in range(5)],
)
plt.title("Transformer test confusion matrix (5x5)")
plt.xlabel("Predicted topic")
plt.ylabel("True topic")
plt.tight_layout()
plt.show()

torch.save(tmodel.state_dict(), DATA_DIR / "models" / "transformer_cls.pt")
print("Saved", DATA_DIR / "models" / "transformer_cls.pt")

# --- Attention heatmaps: 3 correctly classified articles, 2 heads from last layer ---
tmodel.eval()
samples = []
with torch.no_grad():
    for i in range(len(te_y)):
        if len(samples) >= 3:
            break
        ids = te_x[i]
        L = min(ids.numel(), p3.MAX_LEN)
        x1 = ids[:L].unsqueeze(0).to(device)
        lb = torch.tensor([L], device=device)
        logits, attn = tmodel(x1, lb)
        pred = int(logits.argmax(-1).item())
        if pred != te_y[i]:
            continue
        toks = articles_p3[te_aid[i]].split()[:L]
        samples.append((toks, attn.cpu(), pred, te_y[i]))

if len(samples) < 3:
    print("Note: fewer than 3 correctly classified test articles for attention heatmaps.")

for si, (toks, attn, pred, gold) in enumerate(samples):
    heads = [0, min(3, attn.size(1) - 1)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    Tvis = min(len(toks) + 1, attn.size(-1))
    labels = ["[CLS]"] + toks[: Tvis - 1]
    for ax, h in zip(axes, heads):
        w = attn[0, h, 0, :Tvis].numpy()
        ax.imshow(w.reshape(1, -1), aspect="auto", cmap="magma")
        ax.set_yticks([0])
        ax.set_yticklabels([f"Head {h} CLS row"])
        ax.set_xticks(range(Tvis))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
        ax.set_xlabel("Key position (token)")
        ax.set_title(f"Article {si+1} pred={p3.TOPIC_NAMES[pred]} (gold={p3.TOPIC_NAMES[gold]})")
    plt.suptitle("Final encoder layer — CLS attention (two heads)", y=1.12)
    plt.tight_layout()
    plt.show()

# --- BiLSTM vs Transformer (10–15 sentences, Part 3 rubric) ---
best_ep_t = int(np.argmax(hist_t_acc_va)) + 1
best_ep_l = int(np.argmax(hist_l_acc_va)) + 1
mean_t = float(np.mean(trans_epoch_times)) if trans_epoch_times else 0.0
mean_l = float(np.mean(lstm_epoch_times)) if lstm_epoch_times else 0.0
if trans_acc > lstm_acc:
    winner = "Transformer"
elif lstm_acc > trans_acc:
    winner = "BiLSTM topic"
else:
    winner = "(tie)"
margin = abs(trans_acc - lstm_acc)
acc_line = (
    f"1) Accuracy: models tied on test accuracy ({trans_acc:.3f}); macro-F1 {trans_f1:.3f} vs {lstm_f1:.3f}."
    if margin < 1e-6
    else f"1) Accuracy: the {winner} achieved higher test accuracy on this 5-way topic task by about {margin:.3f} in absolute terms "
    f"(Transformer {trans_acc:.3f} vs BiLSTM {lstm_acc:.3f}; macro-F1 {trans_f1:.3f} vs {lstm_f1:.3f})."
)
text = f"""
{acc_line}
2) Convergence: the best validation-accuracy epoch was around epoch {best_ep_t} for the Transformer and {best_ep_l} for the BiLSTM baseline,
   so the shallower recurrent model often plateaus earlier, whereas the Transformer can keep refining with global self-attention across tokens.
3) Speed: mean wall-clock time per epoch was roughly {mean_t:.2f}s for the Transformer and {mean_l:.2f}s for the BiLSTM on this machine;
   per step the Transformer does more matrix work (multi-head attention) while the BiLSTM scans sequentially but with smaller per-token ops.
4) Heatmaps: CLS attention mass tends to concentrate on discriminative content words (entities, policy terms, sports cues) rather than
   generic function words, which matches the intuition that attention reweights evidence for the headline topic.
5) Small data (200–300 articles): a compact BiLSTM with pretrained embeddings is often the safer default because it has fewer parameters
   and inductive bias for local order; a Transformer can still win if regularised and if labels are clean, but it is easier to overfit when
   depth and global mixing exceed the effective sample size.
   Strong data augmentation or label denoising would be prerequisites before scaling width or depth on such a small regime.
   Different heads sometimes show complementary cues (e.g., geography vs institution names), visible as differing attention mass patterns.
"""
print(text)
